# Lesson 17: How Everything Connects

You've been using `python output/chat.py` to create articles. But what happens **behind the scenes** when you say "Create an article about SEO"?

This lesson traces the **full journey of a request** — from the moment you type a message to the moment an article appears in the database. No code to run, no API calls — a guided tour of how the pieces fit together.

## The Journey of a Request

When you type "Create an article about on-page SEO" in the chat, here's what happens:

```
You (chat message)
  │
  ▼
chat.py ─── Team Leader (Opus) decides who should handle it
  │
  ▼
tools/workspace.py ─── Content Creator calls create_article()
  │
  ▼
pipeline.py ─── Runs 4 agent steps: research → outline → write → enrich
  │
  ▼
agents/ ─── Each agent file does its specialized work
  │
  ▼
tools/airtable.py ─── Status and content saved at every step
  │
  ▼
content/ ─── Final article saved as a .md file
```

Each file has a **single responsibility**. One by one:

## chat.py — The Front Door

`chat.py` is the **entry point** — the file you run with `python output/chat.py`.

It does three things:
1. Validates API keys (`ANTHROPIC_API_KEY`, `XAI_API_KEY`)
2. Validates Airtable configuration
3. Imports the team from `agents/team.py` and starts the chat

The team definition lives in `agents/team.py`, and each member is defined in its own file:
- **Content Creator** (`agents/content_creator.py`) — creates articles, retries, batch processing
- **Status Tracker** (`agents/status_tracker.py`) — checks article status, details, history
- **SEO Manager** (`agents/seo_manager.py`) — published URLs, rank tracking
- **Team Leader** (`agents/team.py`) — Claude Opus, reads your message and delegates

Each member has **specific tools** from `tools/workspace.py`. The Leader never calls tools directly — it delegates to the right member.

This follows the **project manager pattern**: you talk to one person, they assign the work.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

# Let's look at the team structure in chat.py
chat_path = os.path.abspath("../../output/chat.py")
with open(chat_path, "r", encoding="utf-8") as f:
    chat_code = f.read()

print(f"chat.py ({len(chat_code.splitlines())} lines)\n")

# Show the team member definitions
for i, line in enumerate(chat_code.splitlines(), 1):
    print(f"  {i:3d} | {line}")

## tools/workspace.py — The Bridge

When the Content Creator receives "Create an article about on-page SEO", it calls the `create_article()` tool.

`tools/workspace.py` bridges the chat team and the pipeline. It contains plain Python functions that:
1. Accept simple parameters (topic, keywords)
2. Call pipeline and Airtable functions
3. Return JSON strings that the AI can interpret

Why a separate bridge? The AI team members need **simple, well-documented functions** as tools. The pipeline has more complex logic. The bridge keeps both sides clean.

In [ ]:
# Let's look at the create_article function in tools/workspace.py
ws_path = os.path.abspath("../../output/tools/workspace.py")
with open(ws_path, "r", encoding="utf-8") as f:
    ws_code = f.read()

# Show just the create_article function (first ~48 lines)
lines = ws_code.splitlines()
print("tools/workspace.py — create_article() function:\n")
for i, line in enumerate(lines[:48], 1):
    print(f"  {i:3d} | {line}")

## pipeline.py — The Engine

`pipeline.py` is the **core** of the product. The `run_content_pipeline()` function runs 4 sequential steps:

```
1. Research  → research_agent searches the web (Claude Sonnet + DuckDuckGo)
2. Outline   → outline_agent creates structure (Claude Sonnet + output_schema)
3. Write     → writer_agent writes the article (Grok-4, plain Markdown)
4. Enrich    → image_agent finds images (Claude Sonnet, optional)
```

Between each step, the pipeline:
- Updates the article status in Airtable
- Catches errors so one failure doesn't crash everything

The status flow:
```
queued → researching → outlining → writing → enriching → review
                                                          ↓
                                                    (manual) published
```

If any step fails: `status → error` with `error_message` saved.

In [ ]:
# Let's look at the pipeline code
pipeline_path = os.path.abspath("../../output/pipeline.py")
with open(pipeline_path, "r", encoding="utf-8") as f:
    pipeline_code = f.read()

print(f"pipeline.py ({len(pipeline_code.splitlines())} lines)\n")

# Show the run_content_pipeline function (the core logic)
lines = pipeline_code.splitlines()
for i, line in enumerate(lines[:161], 1):
    print(f"  {i:3d} | {line}")

## tools/airtable.py — Where Data Lives

`tools/airtable.py` manages the Airtable database (cloud-hosted). It provides these functions:

| Function | Purpose |
|----------|---------|
| `create_article(topic, keywords)` | Insert a new article (status = 'queued') |
| `update_article_status(id, status, **fields)` | Update status + any fields |
| `get_article(id)` | Fetch one article as a dict |
| `list_articles()` | List all articles |

Every function returns **plain Python dicts** — no ORM, no complexity. Article IDs are Airtable record ID strings (like `"recABC123"`).

In [ ]:
# Let's see what an article dict looks like
from tools.airtable import get_article, list_articles

# Show the structure of an article dict
articles = list_articles()
if articles:
    article = articles[0]
    print("An article dict looks like this:\n")
    for key, value in article.items():
        val_preview = str(value)[:60] + "..." if value and len(str(value)) > 60 else value
        print(f"  {key:<20} = {val_preview}")
else:
    print("No articles in Airtable yet.")
    print("An article dict has these keys:")
    print("  id, topic, target_keywords, status, outline_json,")
    print("  article_markdown, output_file, word_count, meta_description,")
    print("  error_message, published_url")

## agents/ — The AI Workers

The `agents/` directory contains one file per agent, plus shared schemas:

| File | Agent | Model | Tools | Output |
|------|-------|-------|-------|--------|
| `researcher.py` | Research Agent | Claude Sonnet | DuckDuckGo search | Plain text (research notes) |
| `outliner.py` | Outline Agent | Claude Sonnet | None | ContentOutline (structured schema) |
| `writer.py` | Writer Agent | Grok-4 | None | Plain Markdown article |
| `image.py` | Image Agent | Claude Sonnet | Freepik/DataForSEO | EnrichedContent (article + images) |
| `content_creator.py` | Content Creator | Claude Sonnet | workspace tools | Chat team member |
| `status_tracker.py` | Status Tracker | Claude Sonnet | workspace tools | Chat team member |
| `seo_manager.py` | SEO Manager | Claude Sonnet | workspace tools | Chat team member |
| `team.py` | Team assembly | Claude Opus (leader) | — | Agno Team |
| `schemas.py` | — | — | — | Pydantic models (ContentOutline, EnrichedContent, etc.) |

**Why different models?**
- Claude Sonnet supports `tools + output_schema` together — good for research and outline
- Grok-4 is used for writing because it produces high-quality long-form content
- The Image Agent is **optional** — if no image API keys are configured, it returns `None` and the pipeline skips enrichment

**One agent = one file.** Want to change the writer? Open `writer.py`. Want to add a fact-checker? Create `fact_checker.py`.

## tools/ — External APIs and Utilities

The `tools/` folder contains integrations and utility modules:

| File | Purpose |
|------|---------|
| `airtable.py` | Primary database — create, read, update articles |
| `workspace.py` | Chat team member tools (bridge to pipeline + Airtable) |
| `rankings.py` | Check keyword positions on Google via DataForSEO |

All of these are designed to **degrade gracefully**:
- No Airtable keys → functions raise clear errors
- No DataForSEO → rank checking returns empty results

Note: Image toolkits (`FreepikTools`, `DataForSEOTools`) are defined in `agents/image.py` alongside the image agent that uses them, not in the `tools/` directory. This keeps each agent's dependencies close to its definition.

## Why This File Structure?

Why not put everything in one file? Or organize by feature? Here's the philosophy:

**One file = one job.** `researcher.py` does research. `writer.py` writes. `airtable.py` talks to Airtable. You never have to guess where something lives.

**One agent = one file.** Want to change the writer's instructions? Open `agents/writer.py`. Want to add a fact-checker agent? Create `agents/fact_checker.py`, import it in `__init__.py`, and wire it into the pipeline. You don't need to understand the whole codebase to make changes.

**Naming = navigation.** File names tell you what's inside without opening them. If someone says "the outline is wrong", you open `agents/outliner.py`. If someone says "articles aren't saving", you look at `tools/airtable.py`.

**`agents/` vs `tools/`** — AI agent definitions vs external service integrations. Agents are the "thinking" layer (they use LLMs). Tools are the "doing" layer (they call APIs, read files, query databases).

**`__init__.py` as a directory** — The `agents/__init__.py` re-exports everything, so `from agents import research_agent` works whether agents is one file or a whole directory. Code that *uses* agents (like `pipeline.py`) doesn't need to change when you reorganize.

## Summary

Everything connects through **plain Python function calls**. No magic.

- **`chat.py`** — Entry point. Validates keys, imports team, starts the chat.
- **`agents/team.py`** — Team Leader (Opus) delegates to 3 Sonnet members.
- **`agents/*.py`** — One file per agent. Pipeline agents + chat team members + schemas.
- **`tools/workspace.py`** — Bridge. Wraps pipeline/Airtable calls as tool functions.
- **`pipeline.py`** — Engine. Runs 4 agents sequentially with Airtable updates at each step.
- **`tools/airtable.py`** — Storage. Airtable cloud database with plain dict functions.
- **`tools/rankings.py`** — SERP rank tracking via DataForSEO.

The full flow: **chat.py → agents/team.py → tools/workspace.py → pipeline.py → agents/*.py → tools/airtable.py**

The philosophy: **one file = one job, one agent = one file, naming = navigation.**

**Next lesson:** The chat interface in action — how workspace tools connect the team to everything you've seen here.

## Exercise

Open `output/pipeline.py` in a text editor (or read it in the code cell above) and answer these questions:

1. Find the 4 agent calls (`research_agent.run(...)`, `outline_agent.run(...)`, etc.). What does each agent receive as input?
2. What function is called between each step to update the database?
3. What happens if the image agent is `None`?
4. Where is the final article file saved? What determines the filename?

Read-only questions — no API calls needed. You're reading code, the same way Claude Code does when it explores a codebase.

<details>
<summary>Click to reveal answers</summary>

1. **Agent inputs:**
   - `research_agent` receives: the topic string
   - `outline_agent` receives: the research notes from step 1
   - `writer_agent` receives: the outline JSON from step 2
   - `image_agent` receives: the article markdown from step 3

2. **Between each step:** `update_article_status(article_id, status, **fields)` is called to save progress to Airtable.

3. **If image agent is None:** The pipeline skips the enrichment step entirely (`if image_agent is not None:`) and uses the writer's output as the final article.

4. **File location:** Saved to the `content/` directory. Filename format: `{slug}-{article_id}-{timestamp}.md` where slug is the topic in lowercase with hyphens.
</details>